# E1：Hyperbolic 表达表示实验

Poincaré 双曲 AE，检验负曲率约束和测地距离是否提供优势。

本 notebook 独立使用一张 GPU。请选用独立 kernel，从第一格顺序运行。默认留出 1/2/3，每折 5 seeds、100 epochs；三组同时运行时使用相同配置。

当前只做表示验证，输出冻结编码器与评价；后续才将得到验证的表示接入 DeepSpatial UOT。详细指标与实验边界见 [README](README.md)。

In [1]:
# 必须在导入 torch 前运行；换卡时重启本 notebook 的 kernel。
import os
import sys
GPU_ID = "5"  # 物理 GPU 编号；调度器已限制可见卡时可设 None，保留分配。
if "torch" in sys.modules:
    raise RuntimeError("当前 kernel 已导入 torch，请重启 kernel 后从第一格运行。")
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "scheduler/default"))


CUDA_VISIBLE_DEVICES = 5


In [2]:
from pathlib import Path
from copy import deepcopy
# 支持从仓库根目录、MMDeepSpatial 或其子目录启动；也支持仅上传 MMDeepSpatial。
PROJECT = next((candidate for parent in (Path.cwd(), *Path.cwd().parents)
                for candidate in (parent, parent / "MMDeepSpatial")
                if (candidate / "mmdeepspatial/experiment.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在 MMDeepSpatial 或其父目录打开 notebook。")
sys.path.insert(0, str(PROJECT))
from mmdeepspatial.experiment import DEFAULT_CONFIG, run_experiment
from mmdeepspatial.data import input_paths
import pandas as pd
print("MMDeepSpatial:", PROJECT)


ModuleNotFoundError: No module named 'geoopt'

## 统一配置

三组的划分、种子、维度、训练和 probe 参数必须保持一致。`log1p` 适用于非负未 log 表达；已 log 或 scaled 的数据统一改 `identity`。首次检查可三组同时改成 holdout `[2]`、seed `[0]`、epochs `2`，正式运行使用全新输出目录。

In [ ]:
ARM = "E1"
DATA_GLOB = str(PROJECT.parent / "data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad")
OUTPUT_ROOT = PROJECT / "outputs/representations_v1"
HOLDOUTS = [1, 2, 3]  # 文件自然排序后的零起始编号，必须为中间切片
SEEDS = [42]
EPOCHS = 100
DIM = 32  # E0=32E；E1=32H；E2=16E+16H；AE 参数量相同
LABEL_KEY = "cell_class"
LAYER = None  # 例如 "counts"；None 读取 X
TRANSFORM = "log1p"  # 已经 log / 标准化的 X 改成 "identity"
E0_METHOD = "ae"  # E0 可另跑 "pca"；E1/E2 忽略此值
DEVICE = "cuda"  # 本 kernel 可见的单卡；CPU 冒烟测试用 "cpu"

CONFIG = deepcopy(DEFAULT_CONFIG)
CONFIG.update(input_glob=DATA_GLOB, output_root=str(OUTPUT_ROOT), holdouts=HOLDOUTS,
              seeds=SEEDS, epochs=EPOCHS, dim=DIM, label_key=LABEL_KEY, layer=LAYER,
              transform=TRANSFORM, e0_method=E0_METHOD, device=DEVICE)
# 其他固定参数均来自 DEFAULT_CONFIG；改动时三个 notebook 同步修改。
# 例如 CONFIG.update(batch_size=512, curvature=1.0, fusion_alpha=0.5)
CONFIG


In [ ]:
paths = input_paths(CONFIG["input_glob"])
display(pd.DataFrame({"slice_index": range(len(paths)), "file": [str(p) for p in paths],
                      "used_as_holdout": [i in HOLDOUTS for i in range(len(paths))]}))
print("总训练任务:", len(HOLDOUTS) * len(SEEDS))
print("数据预处理、编码器和 probe 均在隐藏切片评价前冻结。")


## 训练与评价

各任务自动保存 checkpoint、预处理、嵌入、逐细胞预测、逐类型指标、训练诊断与运行清单。已完成且输入/配置/源码相同的任务可跳过；存在未完成结果时拒绝覆盖，请使用新输出目录。稀有类型按可见训练切片总体丰度 ≤5% 定义，测试未见/缺失类型单列。

In [ ]:
results = run_experiment(ARM, CONFIG)
display(results[["arm", "holdout", "seed", "rare_macro_auprc", "rare_macro_f1",
                 "rare_macro_retrieval_hit_at_5", "rare_macro_knn_purity_at_5",
                 "common_macro_f1", "overall_accuracy", "trainable_parameters"]])


In [ ]:
# 描述性汇总：先在每折内平均 seeds，再展示折间统计。
# 同一组织的留出折有相关性，此标准差不代表独立动物重复的置信区间。
metrics = ["rare_macro_auprc", "rare_macro_f1", "rare_macro_retrieval_hit_at_5",
           "rare_macro_knn_purity_at_5", "rare_macro_coverage", "common_macro_f1"]
by_fold = results.groupby("holdout")[metrics].mean()
display(by_fold)
display(by_fold.agg(["mean", "std"]))
import matplotlib.pyplot as plt
axes = by_fold[["rare_macro_auprc", "rare_macro_f1", "rare_macro_retrieval_hit_at_5"]].plot.bar(
    figsize=(10, 4), ylim=(0, 1), rot=0, title=f"{ARM}: seed-averaged held-out slice metrics")
axes.set_xlabel("Held-out slice index")
plt.tight_layout()
plt.show()


## 三组完成后的配对比较

等 E0/E1/E2 全部完成后，将下一格开关改为 True 再执行。比较器检查协议、输入、预处理、fold/seed 和 AE 参数量，输出 paired delta 与逐类型表。跨切片检索是类型一致性代理，不是真实细胞配对准确率。

In [ ]:
COMPARE_WHEN_READY = False
if COMPARE_WHEN_READY:
    from mmdeepspatial.compare import compare_runs
    all_runs, aggregate, all_types = compare_runs(OUTPUT_ROOT, e0_method=E0_METHOD)
    display(aggregate)
    display(all_types[all_types.is_rare].sort_values(["cell_type", "arm", "holdout", "seed"]))
